In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("Ecommerce").getOrCreate()

# Create df_orders
orders_data = [
    (1, "P001", "U001", "02/25/2023"),
    (2, "P002", "U001", "03/14/2023"),
    (3, "P001", "U002", "03/16/2023"),
    (4, "P003", "U002", "03/18/2023"),
    (5, "P004", "U003", "04/01/2023"),
]

orders_schema = StructType(
    [
        StructField("order_id", IntegerType(), True),
        StructField("product_id", StringType(), True),
        StructField("user_id", StringType(), True),
        StructField("order_date", StringType(), True),
    ]
)

df_orders = spark.createDataFrame(orders_data, schema=orders_schema)

# Create df_products
products_data = [
    ("P001", "Product 1", "Electronics"),
    ("P002", "Product 2", "Clothing"),
    ("P003", "Product 3", "Home Goods"),
    ("P004", "Product 4", "Books"),
]

products_schema = StructType(
    [
        StructField("product_id", StringType(), True),
        StructField("product_name", StringType(), True),
        StructField("category", StringType(), True),
    ]
)

df_products = spark.createDataFrame(products_data, schema=products_schema)

df_orders.show()
df_products.show()

+--------+----------+-------+----------+
|order_id|product_id|user_id|order_date|
+--------+----------+-------+----------+
|       1|      P001|   U001|02/25/2023|
|       2|      P002|   U001|03/14/2023|
|       3|      P001|   U002|03/16/2023|
|       4|      P003|   U002|03/18/2023|
|       5|      P004|   U003|04/01/2023|
+--------+----------+-------+----------+

+----------+------------+-----------+
|product_id|product_name|   category|
+----------+------------+-----------+
|      P001|   Product 1|Electronics|
|      P002|   Product 2|   Clothing|
|      P003|   Product 3| Home Goods|
|      P004|   Product 4|      Books|
+----------+------------+-----------+



In [10]:
df_orders = df_orders.withColumn(
    "is_weekend",
    when(
        lower(date_format(to_date(col("order_date"), "MM/dd/yyyy"), "EEEE")).isin(
            "friday", "saturday", "sunday"
        ),
        lit("1"),
    ).otherwise(lit("0")),
)

In [12]:
df_products.join(df_orders, "product_id", "inner").select(
    "category", "is_weekend", "order_date", "product_name", "user_id"
).show()

+-----------+----------+----------+------------+-------+
|   category|is_weekend|order_date|product_name|user_id|
+-----------+----------+----------+------------+-------+
|Electronics|         1|02/25/2023|   Product 1|   U001|
|Electronics|         0|03/16/2023|   Product 1|   U002|
|   Clothing|         0|03/14/2023|   Product 2|   U001|
| Home Goods|         1|03/18/2023|   Product 3|   U002|
|      Books|         1|04/01/2023|   Product 4|   U003|
+-----------+----------+----------+------------+-------+

